# Part 5 — Across Species: Mouse and Pig Genes to Human

*Written for idtrack 0.1.2.*

This part shows a **humanization** workflow: mapping mouse or pig genes onto human genes, so that you can run human-centred analyses (pathways, marker lists, integration, annotation).

**Learning objectives**
- Understand what humanization is, and when it is appropriate.
- Run a step-by-step mouse → human and pig → human mapping.
- Handle orthology that is not one-to-one explicitly.
- Keep the results in a tidy table with their provenance.

> **Warning:** Orthology is not always one-to-one. This part makes the ambiguity visible and leaves the choice, and its record, to you.

## 5.1 — The workflow

1. **Within the species (idtrack):** convert the mouse or pig identifiers to Ensembl gene ids of that species at a fixed release.
2. **Across species (an orthology table):** map those gene ids to human Ensembl gene ids with an orthology table of the same Ensembl release (Ensembl Compara's homologies, exported from BioMart or downloaded from Ensembl's FTP site). idtrack does not compute orthology; it makes sure that the ids on both sides of the table are the ids of the release the table was made for.
3. **Human names (optional, idtrack):** convert the human gene ids to HGNC symbols (or another human database).

This part does **not** decide the "correct" ortholog in complex families: it keeps the candidates visible and asks you to record the rule you apply.

### 5.1.1 Prerequisites

**Graphs.** The graphs of the organisms you use (Part 2): `mus_musculus` and/or `sus_scrofa`, and `homo_sapiens`.

**An orthology table.** Ensembl publishes the orthologs of every release (Ensembl Compara): in BioMart (the attributes "Human gene stable ID" and "Human homology type" of a mouse or pig gene dataset) and as files under `https://ftp.ensembl.org/pub/release-<release>/tsv/ensembl-compara/homologies/`. Use the table of the release you convert to, so that its gene ids are the ids idtrack gives.

In [1]:
# The orthology table of this tutorial: the rows of Ensembl Compara's homologies for human TP53 (mouse Trp53 and pig TP53,
# both one-to-one orthologs), with the columns of a BioMart export (gene stable ID, human gene stable ID, human homology
# type). In your work, read the export of your release, e.g. pd.read_csv('mouse_human_orthologs_115.tsv', sep='\t').
from __future__ import annotations

import pandas as pd

orthology = pd.DataFrame(
    [
        {'source_gene': 'ENSMUSG00000059552', 'human_gene': 'ENSG00000141510', 'homology_type': 'ortholog_one2one'},
        {'source_gene': 'ENSSSCG00000017950', 'human_gene': 'ENSG00000141510', 'homology_type': 'ortholog_one2one'},
    ]
)
orthology

,source_gene,human_gene,homology_type
0,ENSMUSG00000059552,ENSG00000141510,ortholog_one2one
1,ENSSSCG00000017950,ENSG00000141510,ortholog_one2one


## 5.2 — Mouse → human, step by step

### 5.2.1 A mouse identifier to a mouse Ensembl gene id (idtrack)

Start with what you have (often an MGI symbol) and convert it to a **base Ensembl gene id** (without version) in mouse.

In [2]:
import os
from pathlib import Path

from idtrack import API

# The local repository (IDTRACK_LOCAL_REPO is a convention of these tutorials; idtrack reads no such variable).
LOCAL_REPOSITORY = Path(os.environ.get('IDTRACK_LOCAL_REPO', 'idtrack_cache'))
LOCAL_REPOSITORY.mkdir(parents=True, exist_ok=True)

API.configure_logger()

RELEASE = 115  # the release of the graphs and of the orthology table
api_mouse = API.build('mus_musculus', RELEASE, str(LOCAL_REPOSITORY))

mouse_query = 'Trp53'  # an MGI symbol; replace with your gene
mouse_to_ensembl = api_mouse.convert(mouse_query, to_release=RELEASE, final_database='base_ensembl_gene')
print(mouse_to_ensembl.status, list(mouse_to_ensembl.targets()))

2026-11-24 11:52:28 WARNING:idtrack.build.externals: No externals YAML in the local repository (`idtrack_cache/mus_musculus_externals_modified.yml`); the package uses its default file for mus_musculus (`.../idtrack/default_config/mus_musculus_externals_modified.yml`).


converted ['ENSMUSG00000059552']


`targets()` holds the mouse Ensembl gene id (without version, because of `final_database='base_ensembl_gene'`). If there are several, the answer is a tie (`tie=True`: genes equal in all evidence); their order is fixed (by gene id) and is not a choice by evidence, so decide how to handle it instead of taking the first.

### 5.2.2 The human ortholog(s) (the orthology table)

Look the mouse gene up in the orthology table of the same release, and keep every ortholog it lists, with its type.

In [3]:
mouse_targets = mouse_to_ensembl.targets()
mouse_ensembl_gene_id = mouse_targets[0] if len(mouse_targets) == 1 else None

if mouse_ensembl_gene_id is None:
    print('No single mouse Ensembl gene id; look at the result (status, tie) first.')
    human_ensembl_gene_ids = []
else:
    rows = orthology[orthology['source_gene'] == mouse_ensembl_gene_id]
    print(rows.to_string(index=False))
    human_ensembl_gene_ids = sorted(rows['human_gene'])

human_ensembl_gene_ids

       source_gene      human_gene    homology_type
ENSMUSG00000059552 ENSG00000141510 ortholog_one2one


['ENSG00000141510']

### 5.2.3 Human gene ids to HGNC symbols (idtrack)

Now the human graph converts the human gene ids to HGNC symbols (optional, but common).

In [4]:
api_human = API.build('homo_sapiens', RELEASE, str(LOCAL_REPOSITORY))

# Convert every candidate ortholog (a many-to-many case shows here)
human_results = api_human.convert_many(human_ensembl_gene_ids, to_release=RELEASE, final_database='HGNC Symbol')
for r in human_results:
    print(r.query_id, '->', list(r.targets()), '| tie:', r.tie)

2026-11-24 11:52:31 WARNING:idtrack.build.externals: No externals YAML in the local repository (`idtrack_cache/homo_sapiens_externals_modified.yml`); the package uses its default file for homo_sapiens (`.../idtrack/default_config/homo_sapiens_externals_modified.yml`).


ENSG00000141510 -> ['TP53'] | tie: False


## 5.3 — Pig → human (the same pattern)

Use the pig graph and start from your pig identifiers: Ensembl ids, NCBI gene ids or VGNC symbols (`final_database` names what you want back).

In [5]:
api_pig = API.build('sus_scrofa', RELEASE, str(LOCAL_REPOSITORY))

pig_query = 'TP53'  # a pig symbol (VGNC); replace with your gene symbol or id
pig_to_ensembl = api_pig.convert(pig_query, to_release=RELEASE, final_database='base_ensembl_gene')
print(pig_to_ensembl.status, list(pig_to_ensembl.targets()))
print(orthology[orthology['source_gene'].isin(pig_to_ensembl.targets())].to_string(index=False))

2026-11-24 11:52:36 WARNING:idtrack.build.externals: No externals YAML in the local repository (`idtrack_cache/sus_scrofa_externals_modified.yml`); the package uses its default file for sus_scrofa (`.../idtrack/default_config/sus_scrofa_externals_modified.yml`).


converted ['ENSSSCG00000017950']
       source_gene      human_gene    homology_type
ENSSSCG00000017950 ENSG00000141510 ortholog_one2one


From here, the orthology lookup and the human conversion are those of 5.2.2 and 5.2.3.

## 5.4 — Several ortholog candidates

When the table lists several orthologs (`ortholog_one2many`, `ortholog_many2many`), choose with the evidence the orthology resource gives (Ensembl Compara reports the percent identity of each pair, the gene-order conservation and whether the ortholog is high-confidence), and record the rule you used; idtrack keeps every candidate visible but does not rank orthologs.

In [6]:
# Keep every ortholog, and the rule that would pick one, as columns of the result
candidates = orthology.groupby('source_gene')['human_gene'].apply(list)
candidates.rename('human_candidates').to_frame().assign(rule='one2one only: keep; otherwise review')

,human_candidates,rule
source_gene,,
ENSMUSG00000059552,[ENSG00000141510],one2one only: keep; otherwise review
ENSSSCG00000017950,[ENSG00000141510],one2one only: keep; otherwise review


### Practical cautions

1. **Orthology depends on context**: paralogs, expansions of gene families and differences of annotation matter.
2. **Do not silently pick one** in many-to-many cases; record the rule you used.
3. **Record the provenance**: the releases of the graphs, their externals configurations if you changed them, and the orthology resource and its release.

## 5.5 — A table to keep

Once the genes are humanized, keep the result in a form you can reproduce and check:

1. **Within each species:** Ensembl gene ids at a fixed release.
2. **Across species:** the human orthologs, with the ambiguity kept (1→n mappings stored as lists).
3. **Provenance:** releases, assemblies, and the orthology source.
4. **A policy for ambiguous cases:** drop, keep all, or choose (and say why).

A practical format is a tidy table with one row per input gene and explicit columns of provenance. The cell below makes one from the results above.

In [7]:
def humanize(api_source, source_id, organism):
    """One row of the table: a source gene, its human orthologs from the table, and their HGNC symbols."""
    source = api_source.convert(source_id, to_release=RELEASE, final_database='base_ensembl_gene')
    genes = list(source.targets())
    human = sorted(orthology.loc[orthology['source_gene'].isin(genes), 'human_gene'])
    symbols = [list(r.targets()) for r in api_human.convert_many(human, to_release=RELEASE, final_database='HGNC Symbol')]
    return {
        'source_species': organism,
        'source_id': source_id,
        'source_status': source.status,
        'source_ensembl_genes': genes,
        'human_ensembl_genes': human,
        'human_hgnc_symbols': symbols,
        'release': RELEASE,
        'orthology_source': 'Ensembl Compara homologies (BioMart export)',
    }


mapping_table = pd.DataFrame(
    [
        humanize(api_mouse, 'Trp53', 'mus_musculus'),
        humanize(api_pig, 'TP53', 'sus_scrofa'),
        humanize(api_mouse, 'Not_a_gene', 'mus_musculus'),  # a 1→0 row: keep such rows for reporting
    ]
)
mapping_table

,source_species,source_id,source_status,source_ensembl_genes,human_ensembl_genes,human_hgnc_symbols,release,orthology_source
0,mus_musculus,Trp53,converted,[ENSMUSG00000059552],[ENSG00000141510],[[TP53]],115,Ensembl Compara homologies (BioMart export)
1,sus_scrofa,TP53,converted,[ENSSSCG00000017950],[ENSG00000141510],[[TP53]],115,Ensembl Compara homologies (BioMart export)
2,mus_musculus,Not_a_gene,not_found,[],[],[],115,Ensembl Compara homologies (BioMart export)
